In [2]:
import sys
import csv
import logging
import time
import re
import os
from datetime import datetime, timedelta

import pandas as pd
from dateutil.relativedelta import relativedelta

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager

# Configuration du logger
logging.basicConfig(
    level=logging.INFO,  # Passez en DEBUG pour voir les messages détaillés
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Fonction utilitaire pour afficher des messages de débogage détaillés.
def detailed_debug(context: str, label: str, value):
    logging.debug(f"[{context}] {label}: {repr(value)}")

# Décorateur pour tracer l'entrée et la sortie d'une fonction
def debug_decorator(func):
    def wrapper(*args, **kwargs):
        logging.debug(f"Entrée dans {func.__name__} avec args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        logging.debug(f"Sortie de {func.__name__} avec résultat={result}")
        return result
    return wrapper

In [3]:
# Instanciation du service Firefox avec GeckoDriverManager
sele = FirefoxService(GeckoDriverManager().install())

2025-03-23 15:09:56,370 - INFO - ====== WebDriver manager ======
2025-03-23 15:09:56,656 - INFO - Get LATEST geckodriver version for 136.0 firefox
2025-03-23 15:09:56,877 - INFO - Get LATEST geckodriver version for 136.0 firefox
2025-03-23 15:09:56,957 - INFO - Driver [/Users/magellan/.wdm/drivers/geckodriver/mac64/v0.36.0/geckodriver] found in cache


In [4]:
@debug_decorator
def parse_pbk6be_divs(divs):
    """
    Extrait les informations des <div class="PBK6be"> et renvoie un dictionnaire.
    """
    data = {}
    for div in divs:
        full_text = div.get_text(strip=True)
        detailed_debug("parse_pbk6be_divs", "full_text", full_text)
        
        # Cas 1 : format avec <b> (ex: "Cuisine : 5")
        b_elem = div.find("b")
        if b_elem:
            match = re.search(r"(Cuisine|Service|Ambiance)\s*:\s*(\d+)", b_elem.parent.get_text(strip=True))
            if match:
                label = match.group(1)
                score = match.group(2)
                detailed_debug("parse_pbk6be_divs", "match", {label: score})
                if label == "Service":
                    data["Service_score"] = score
                else:
                    data[label] = score
                continue

        # Cas 2 : format "LABEL : VALEUR" via span en gras
        bold_span = div.find("span", style=re.compile(r"font-weight:\s*bold"))
        if bold_span:
            label_text = bold_span.get_text(strip=True)
            remainder = full_text.replace(label_text, "", 1).strip()
            remainder = re.sub(r'\s+', ' ', remainder.replace("\n", ""))
            detailed_debug("parse_pbk6be_divs", "label_text/remainder", {label_text: remainder})
            data[label_text] = remainder
            continue

        # D'autres règles de parsing peuvent être ajoutées ici si nécessaire.
    return data


@debug_decorator
def transform_date(df):
    """
    Transforme la colonne 'Review Time' et convertit la date de collecte en datetime.
    """
    df["Review Time"] = df["Review Time"].apply(lambda x: x.strip("il y a "))
    df["Review Time"] = df["Review Time"].str.replace("une", "1")
    df["Review Time"] = df["Review Time"].str.replace("un", "1")
    df["Review Time"] = df["Review Time"].str.replace("\xa0", " ")
    df["Review Time"] = df["Review Time"].str.replace("ans", "an")
    df["Review Time"] = df["Review Time"].str.replace("an", "ans")
    df["Review Time"] = df["Review Time"].str.replace("jours", "jour")
    df["Review Time"] = df["Review Time"].str.replace("jour", "jours")
    df["Review Time"] = df["Review Time"].str.replace("semaine", "semaines")
    df["Review Time"] = df["Review Time"].str.replace("semaines", "semaine")
    df["Review date collected"] = pd.to_datetime(df["Review date collected"])
    detailed_debug("transform_date", "Transformed DataFrame", df.head())
    return df


@debug_decorator
def estimated_date(google_date, collected_date):
    """
    Calcule la date estimée de l'avis à partir de la chaîne 'google_date' et la date de collecte.
    """
    try:
        nunits, units = google_date.split(" ", 1)
        nunits = int(nunits)
        detailed_debug("estimated_date", "Parsed", {"nunits": nunits, "units": units})
    except Exception as e:
        logging.warning(f"Erreur lors du parsing de google_date: {e}")
        return collected_date

    if units in ["minute", "minutes"]:
        result = collected_date - relativedelta(minutes=nunits)
    elif units in ["heure", "heures"]:
        result = collected_date - relativedelta(hours=nunits)
    elif units in ["jour", "jours"]:
        result = collected_date - relativedelta(days=nunits)
    elif units in ["semaine", "semaines"]:
        result = collected_date - relativedelta(weeks=nunits)
    elif units == "mois":
        result = collected_date - relativedelta(months=nunits)
    elif units == "ans":
        result = collected_date - relativedelta(years=nunits)
    else:
        result = collected_date
    detailed_debug("estimated_date", "result", result)
    return result

In [5]:
@debug_decorator
def rec_log(entreprise, name, url, nb_avis_disponible, delta=None):
    """
    Enregistre les informations de log dans un fichier CSV.
    """
    logging.info(f"rec_log() - entreprise={entreprise}, name={name}, url={url}, nb_avis_disponible={nb_avis_disponible}, delta={delta}")
    date_execution = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_data = {
        'entreprise': [entreprise],
        'name': [name],
        'url': [url],
        'nb_avis': [nb_avis_disponible],
        'delta': [delta],
        'date': [date_execution]
    }
    new_df = pd.DataFrame(log_data)
    fichier_existe = os.path.isfile('log.csv')
    detailed_debug("rec_log", "fichier_existe", fichier_existe)
    if fichier_existe:
        existing_df = pd.read_csv('log.csv')
        updated_df = pd.concat([existing_df, new_df], ignore_index=True)
        updated_df.to_csv('log.csv', index=False)
    else:
        new_df.to_csv('log.csv', index=False)
    print('Le log a été enregistré avec succès.')
    logging.info("Le log a bien été enregistré dans log.csv")


@debug_decorator
def get_review_summary(result_set):
    """
    Construit un DataFrame résumant les avis extraits de la page.
    """
    rev_dict = {
        'Review Rate': [],
        'Review Service': [],
        'Review Ambiance': [],
        'Review Service_score': [],
        'Review Cuisine': [],
        'Review Type de repas': [],
        'Review Plats recommandés': [],
        'Review Time': [],
        'Review Text': [],
        'Review date collected': []
    }

    for idx, result in enumerate(result_set, start=1):
        try:
            review_rate = result.find('span', class_='kvMYJc')['aria-label']
        except Exception as e:
            logging.error(f"Erreur lors de l'extraction du 'Review Rate' : {e}")
            review_rate = "N/A"

        review_time = result.find('span', class_='rsqaWe').text
        try:
            review_text = result.find('span', class_='wiI7pd').text
        except Exception:
            review_text = ""

        try:
            review_subitem_raw = result.find_all("div", class_="PBK6be")
            tmp = parse_pbk6be_divs(review_subitem_raw)
            review_service = tmp.get("Service", "")
            review_cuisine = tmp.get("Cuisine", "")
            review_service_score = tmp.get("Service_score", "")
            review_ambiance = tmp.get("Ambiance", "")
            review_type = tmp.get("Type de repas", "")
            review_recommandation = tmp.get("Plats recommandés", "")
        except Exception as e:
            logging.error(f"Erreur lors de l'extraction des sous-éléments : {e}")
            review_service = review_cuisine = review_service_score = review_ambiance = review_type = review_recommandation = ""

        rev_dict['Review Plats recommandés'].append(review_recommandation)
        rev_dict['Review Type de repas'].append(review_type)
        rev_dict["Review Service"].append(review_service)
        rev_dict["Review Ambiance"].append(review_ambiance)
        rev_dict["Review Service_score"].append(review_service_score)
        rev_dict["Review Cuisine"].append(review_cuisine)
        rev_dict['Review Rate'].append(review_rate)
        rev_dict['Review Time'].append(review_time)
        rev_dict['Review Text'].append(review_text)
        rev_dict['Review date collected'].append(datetime.today().strftime("%Y-%m-%d %H:%M:%S"))

    df_reviews = pd.DataFrame(rev_dict)
    detailed_debug("get_review_summary", "DataFrame head", df_reviews.head())
    return df_reviews